# 🔬 Your Research Project
### EPS Research High-School Exploration Track — Ages 15-18

**Project:** Does omega correlate with galaxy size or rotation speed?

Using the same frozen 84-galaxy SPARC cohort from the published analysis,
we compare the published omega values with two directly measured properties:

1. outermost measured radius, $R_{max}$
2. maximum observed rotation speed, $V_{max}$

Correlation does not establish causation; this is an exploratory descriptive test.

**Prerequisites:** Correlation, hypothesis testing, scientific reasoning

In [ ]:
# ── Colab setup: canonical FAIR² corpus paths ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import urllib.request
    CORPORA = {
        'rotation_curve_corpus_v7.json': 'https://zenodo.org/records/19563417/files/rotation_curve_corpus_v7.json',
        'high_z_kinematic_corpus_Z1.json': 'https://zenodo.org/records/21834678/files/high_z_kinematic_corpus_Z1.json',
        'dwarf_irregular_corpus_v1.json': 'https://zenodo.org/records/20320362/files/dwarf_irregular_corpus_v1.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")

    HI_PATH = 'rotation_curve_corpus_v7.json'
    Z1_PATH = 'high_z_kinematic_corpus_Z1.json'
    DWARF_PATH = 'dwarf_irregular_corpus_v1.json'
    print("Ready.")
else:
    HI_PATH = '../hi/rotation_curve_corpus_v7.json'
    Z1_PATH = '../highz/high_z_kinematic_corpus_Z1.json'
    DWARF_PATH = '../dwarfs/dwarf_irregular_corpus_v1.json'
    print("Running locally — using canonical repository corpus paths.")


In [ ]:
import matplotlib
matplotlib.use('Agg')
import json, numpy as np, matplotlib.pyplot as plt

# Frozen published cohort: (canonical SPARC galaxy ID, omega in rad/Gyr)
FROZEN_84 = [('NGC3741', 7.032),
 ('UGC08550', 9.317),
 ('NGC3109', 10.244),
 ('UGC07603', 14.2),
 ('DDO064', 15.35),
 ('UGC01281', 11.37),
 ('UGC07151', 12.53),
 ('UGC07399', 14.86),
 ('UGC04278', 13.76),
 ('NGC3972', 13.93),
 ('NGC7793', 11.4),
 ('F571-8', 9.17),
 ('UGC05721', 11.51),
 ('UGC07323', 13.41),
 ('NGC3521', 10.25),
 ('F563-V2', 11.08),
 ('ESO116-G012', 11.02),
 ('F568-1', 10.52),
 ('ESO079-G014', 10.37),
 ('NGC3893', 6.47),
 ('UGC08286', 9.82),
 ('NGC0024', 9.55),
 ('NGC0100', 9.37),
 ('NGC0891', 8.99),
 ('NGC4217', 9.99),
 ('UGC06917', 8.3),
 ('NGC7814', 8.52),
 ('NGC3917', 8.82),
 ('F583-4', 9.33),
 ('IC4202', 9.3),
 ('UGC08490', 6.95),
 ('NGC4088', 6.98),
 ('NGC6946', 6.83),
 ('F568-3', 6.54),
 ('F568-V1', 6.55),
 ('NGC2403', 6.32),
 ('UGC12632', 6.27),
 ('UGC11455', 6.24),
 ('NGC5985', 6.16),
 ('UGC00731', 5.99),
 ('UGC06786', 5.87),
 ('NGC6195', 5.83),
 ('UGC12732', 5.81),
 ('NGC4157', 5.46),
 ('UGC06930', 5.44),
 ('UGC03205', 5.27),
 ('UGC11820', 5.21),
 ('NGC4100', 5.2),
 ('UGC03546', 5.29),
 ('F563-1', 5.02),
 ('NGC4559', 5.3),
 ('F583-1', 5.16),
 ('NGC2955', 5.71),
 ('NGC7331', 4.9),
 ('NGC4183', 4.92),
 ('DDO161', 4.69),
 ('NGC6503', 4.3),
 ('NGC2998', 4.61),
 ('NGC1090', 5.24),
 ('NGC5033', 3.79),
 ('NGC5371', 3.78),
 ('F579-V1', 7.13),
 ('UGC06983', 5.74),
 ('NGC2841', 3.58),
 ('UGC05750', 3.44),
 ('UGC05005', 3.38),
 ('UGC02885', 3.4),
 ('NGC5055', 2.89),
 ('NGC6674', 2.81),
 ('UGC01230', 2.74),
 ('UGC06614', 2.49),
 ('UGC02487', 2.49),
 ('UGC00128', 2.23),
 ('NGC0801', 3.32),
 ('UGC09133', 1.97),
 ('UGC07125', 3.09),
 ('NGC1003', 3.49),
 ('NGC3198', 3.33),
 ('NGC2903', 7.01),
 ('ESO563-G021', 7.31),
 ('NGC5585', 8.06),
 ('F574-1', 7.68),
 ('UGC06446', 7.11),
 ('UGC07524', 7.15)]

assert len(FROZEN_84) == 84
assert len({name for name,omega in FROZEN_84}) == 84

with open(HI_PATH) as f:
    corpus=json.load(f)

sparc={
    g['galaxy']:g
    for g in corpus['galaxies']
    if g.get('survey')=='SPARC'
}

results=[]
missing=[]

for name,omega_rad_gyr in FROZEN_84:
    g=sparc.get(name)
    if g is None or not g.get('data'):
        missing.append(name)
        continue

    data=[
        p for p in g['data']
        if p.get('Rad',0)>0 and p.get('Vobs',0)>0
    ]
    if len(data)<2:
        missing.append(name)
        continue

    results.append({
        'galaxy':name,
        'omega':omega_rad_gyr,
        'r_max':data[-1]['Rad'],
        'v_max':max(p['Vobs'] for p in data),
    })

assert not missing, f"Frozen cohort missing/invalid: {missing}"
assert len(results)==84

om_arr=np.array([r['omega'] for r in results])
rm_arr=np.array([r['r_max'] for r in results])
vm_arr=np.array([r['v_max'] for r in results])

r_om_rm=np.corrcoef(om_arr,rm_arr)[0,1]
r_om_vm=np.corrcoef(om_arr,vm_arr)[0,1]

fig,axes=plt.subplots(1,2,figsize=(12,5))

ax=axes[0]
ax.scatter(rm_arr,om_arr,s=20,alpha=0.6)
ax.axhline(np.mean(om_arr),lw=1.5,ls='--',
           label=f'Mean ω = {np.mean(om_arr):+.2f}')
ax.set_xlabel('R_max (kpc)',fontsize=12)
ax.set_ylabel('ω (rad/Gyr)',fontsize=12)
ax.set_title(
    f'Omega vs Outer Radius\\nCorrelation r = {r_om_rm:.3f}',
    fontsize=11
)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

ax2=axes[1]
ax2.scatter(vm_arr,om_arr,s=20,alpha=0.6)
ax2.axhline(np.mean(om_arr),lw=1.5,ls='--',
            label=f'Mean ω = {np.mean(om_arr):+.2f}')
ax2.set_xlabel('Vmax (km/s)',fontsize=12)
ax2.set_ylabel('ω (rad/Gyr)',fontsize=12)
ax2.set_title(
    f'Omega vs Max Speed\\nCorrelation r = {r_om_vm:.3f}',
    fontsize=11
)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.suptitle(
    '🔬 Student Research Project — Frozen N=84 SPARC Cohort\\n'
    'Does omega correlate with galaxy size or rotation speed?',
    fontsize=12
)
plt.tight_layout()
plt.savefig('hs_b_09_research_project.png',dpi=150,bbox_inches='tight')
plt.show()

print(f'Sample size: {len(results)} galaxies')
print(f'Omega vs R_max correlation: r = {r_om_rm:.3f}')
print(f'Omega vs Vmax correlation:  r = {r_om_vm:.3f}')
print()
print('These are descriptive correlations in the frozen published cohort.')
